# MMTFv3 CL+GC Optuna — Continuous Position Sizing

Trains **MMTFv3Core** (Mamba or Transformer backbone, Optuna-selected) across CL, GC
using `ContinuousTradingLoss` + `SharpeScheduler` for continuous position sizing ∈ [-1, 1].

## Model: MMTFv3Core
6-phase architecture:
1. **VAE Regime Encoder**: Daily [ret_1d, ret_5d, ret_21d, rv_1d] → z_regime
2. **Static Context**: Identity + z_regime → c_s, c_e, c_c, c_h
3. **Technical Backbone**: ContinuousIntradayPrep features (incl. tod_sin) → Mamba/Transformer
4. **Cross-Modal Branches**: Spatial (NumberBars + VPIN raster) + Sequential (tabular VPIN)
5. **Regime-Conditioned BVS**: Weighted branch selection
6. **Enrichment + Temporal Attention**: TFT-style attention → continuous position head

## Loss: ContinuousTradingLoss
```
total = loss_sharpe + direction_weight * loss_direction + loss_tc + loss_reg
```
With `SharpeScheduler` shifting direction → Sharpe emphasis over training.

## Target
30-minute forward log return from 5-min intraday bars.

## 1. Environment Setup

In [ ]:
# Mount Google Drive (Colab only)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print("Not running in Colab - skipping drive mount")

In [ ]:
# Add CTAFlow to path (Colab only)
if IN_COLAB:
    import sys
    %cd /content/drive/MyDrive/CTAEnv/
    %cd CTAFlow
    !git pull
    %cd ..
    sys.path.insert(0, '/content/drive/MyDrive/CTAEnv/CTAFlow/')
    sys.path.insert(1, '/content/drive/MyDrive/CTAEnv/SierraPy')
    !pip install -e SierraPy -q
    !pip install -e CTAFlow -q
    !pip install optuna -q
else:
    print("Running locally - ensure CTAFlow and optuna are installed")

In [ ]:
import json
import math
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim

import optuna
from optuna.pruners import MedianPruner
from optuna.samplers import TPESampler

warnings.filterwarnings('ignore')

print(f"PyTorch version: {torch.__version__}")
print(f"Optuna version: {optuna.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

def set_seed(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

## 2. Configuration

In [ ]:
# --- Tickers ---
TICKERS = ['CL', 'GC']

# --- Architecture ---
TUNE_BACKBONE = True     # Let Optuna choose mamba/transformer
BACKBONE = 'mamba'       # Default if TUNE_BACKBONE=False

# --- Target ---
TARGET_HORIZON_MINUTES = 30   # 30min forward return
BAR_MINUTES = 5               # 5min bars → target = y_fwd_6

# --- Session Filter ---
# Options:
#   SAMPLE_SESSION = "usa"                     # Named: "usa", "london", "overlap"
#   SAMPLE_SESSION = None                      # Use all active session bars
#   SAMPLE_SESSION_START/END = "09:30"/"15:00" # Custom time window
SAMPLE_SESSION = "usa"              # Filter to USA session only
SAMPLE_SESSION_START = None         # Custom start (overrides SAMPLE_SESSION if both set)
SAMPLE_SESSION_END = None           # Custom end

# --- Stride ---
# stride=6 with 5min bars + 30min target → non-overlapping samples
# stride=3 → 50% overlap, stride=1 → every bar (max overlap)
SAMPLE_STRIDE = 6

# --- AE ---
AE_WINDOW = 21                # 21 trading days lookback for regime VAE
F_AE = 4                      # [ret_1d, ret_5d, ret_21d, rv_1d]

# --- Paths ---
if IN_COLAB:
    DRIVE_PATH = Path('/content/drive/MyDrive')
    DATA_ROOT = DRIVE_PATH / 'features'
    RESULTS_PATH = DRIVE_PATH / 'results' / 'mmtfv3_cl_gc'
else:
    DATA_ROOT = Path('/workspace/model_data')
    RESULTS_PATH = Path('/workspace/results/mmtfv3_cl_gc')

RESULTS_PATH.mkdir(parents=True, exist_ok=True)

print(f"Data root: {DATA_ROOT}")
print(f"Results path: {RESULTS_PATH}")
print(f"Tickers: {TICKERS}")
print(f"Target: {TARGET_HORIZON_MINUTES}min forward return")
if SAMPLE_SESSION_START and SAMPLE_SESSION_END:
    print(f"Session filter: custom {SAMPLE_SESSION_START}-{SAMPLE_SESSION_END}")
elif SAMPLE_SESSION:
    print(f"Session filter: {SAMPLE_SESSION.upper()}")
else:
    print(f"Session filter: all active bars")
print(f"Stride: {SAMPLE_STRIDE} bars ({SAMPLE_STRIDE * BAR_MINUTES}min between samples)")
print(f"AE window: {AE_WINDOW} days, f_ae={F_AE}")
if TUNE_BACKBONE:
    print(f"Backbone: Optuna-tuned (mamba / transformer)")
else:
    print(f"Backbone: {BACKBONE.upper()} (fixed)")

In [ ]:
# Verify data files for each ticker
print("Checking data files...")
required_files = ['intraday.csv', 'profiles.npz', 'rasterized.npz', 'vpin.parquet']

all_found = True
for ticker in TICKERS:
    ticker_path = DATA_ROOT / ticker
    print(f"\n{ticker}:")
    for fname in required_files:
        fpath = ticker_path / fname
        status = "[OK]" if fpath.exists() else "[MISSING]"
        print(f"  {status} {fname}")
        if not fpath.exists():
            all_found = False

if not all_found:
    print("\n[WARNING] Some files missing - data loading may fail")

## 3. Load Data via V3ContinuousPrep

In [ ]:
from CTAFlow.data.datasets.v3_continuous import (
    V3ContinuousPrep,
    V3ContinuousDataset,
    v3_collate_fn,
    unpack_v3_batch,
    build_v3_loaders,
)
from CTAFlow.models.prep.intraday_continuous import SessionSpec

print("Loading ticker data...")
prep = V3ContinuousPrep.from_directories(
    root_dir=DATA_ROOT,
    tickers=TICKERS,
    sessions=[SessionSpec("USA", "08:30", "16:00")],
    bar_minutes=BAR_MINUTES,
    target_horizon_minutes=TARGET_HORIZON_MINUTES,
    ae_window=AE_WINDOW,
)

dims = prep.get_dims()
print(f"\nFeature dimensions: {dims}")
print(f"Tech feature columns ({dims['f_tech']}): {prep._tech_feature_cols[:10]}...")
print(f"n_tickers: {prep.n_tickers}")
print(f"n_asset_classes: {prep.n_asset_classes}")
print(f"n_asset_subclasses: {prep.n_asset_subclasses}")

In [ ]:
# Verify tech features include tod_sin
assert 'tod_sin' in prep._tech_feature_cols, "tod_sin missing from feature cols!"
print("tod_sin confirmed in feature columns")

# Quick target distribution check
fig, axes = plt.subplots(1, len(TICKERS), figsize=(6*len(TICKERS), 4))
if len(TICKERS) == 1:
    axes = [axes]

for ticker, ax in zip(TICKERS, axes):
    df = prep._tech_dfs.get(ticker)
    if df is None:
        continue
    target = df[prep._target_col].dropna()
    ax.hist(target.values, bins=100, alpha=0.7, color='steelblue')
    ax.axvline(x=0, color='red', linestyle='--', alpha=0.5)
    ax.set_title(f'{ticker} Target ({prep._target_col})')
    ax.set_xlabel('30min Forward Log Return')
    ax.set_ylabel('Count')
    print(f"{ticker}: mean={target.mean():.6f}, std={target.std():.6f}, n={len(target)}")

plt.tight_layout()
plt.show()

## 4. Define Optuna Objective

In [ ]:
from CTAFlow.models.deep_learning.multi_branch.tft.mmtf_v3_core import (
    MMTFv3Core,
    MMTFv3Mamba,
    MMTFv3Transformer,
    train_epoch_v3,
    evaluate_v3,
    ContinuousTradingLoss,
    SharpeScheduler,
)

F_TECH = dims['f_tech']
F_SEQ = dims['f_seq']

print(f"Model input dimensions:")
print(f"  f_tech={F_TECH} (ContinuousIntradayPrep features incl. tod_sin)")
print(f"  f_seq={F_SEQ} (tabular VPIN)")
print(f"  f_ae={F_AE} (daily returns: ret_1d, ret_5d, ret_21d, rv_1d)")
print(f"  n_tickers={prep.n_tickers}, n_asset_classes={prep.n_asset_classes}, n_subclasses={prep.n_asset_subclasses}")
print(f"\nArchitecture: MMTFv3Core — 6-phase continuous position sizing")
print(f"Loss: ContinuousTradingLoss + SharpeScheduler")

In [ ]:
def objective(trial: optuna.Trial) -> float:
    # --- Backbone selection ---
    if TUNE_BACKBONE:
        backbone = trial.suggest_categorical('backbone', ['mamba', 'transformer'])
    else:
        backbone = BACKBONE

    # --- Shared architecture ---
    d_model = trial.suggest_categorical('d_model', [64, 128])
    d_static_emb = trial.suggest_categorical('d_static_emb', [32, 64])
    n_heads = trial.suggest_categorical('n_heads', [2, 4])
    dropout = trial.suggest_float('dropout', 0.1, 0.4)
    grn_dropout = trial.suggest_float('grn_dropout', 0.05, 0.4)

    # --- Backbone-specific ---
    if backbone == 'mamba':
        d_state = trial.suggest_categorical('d_state', [16, 32])
        d_conv = trial.suggest_categorical('d_conv', [2, 4])
        expand = trial.suggest_categorical('expand', [1, 2])
        n_layers = trial.suggest_int('n_layers', 1, 2)
        d_ff = 512
    else:
        n_layers = trial.suggest_int('n_layers', 2, 4)
        d_ff = trial.suggest_categorical('d_ff', [256, 512])
        d_state, d_conv, expand = 16, 4, 2

    # --- VAE ---
    ae_type = trial.suggest_categorical('ae_type', ['vae', 'vqvae'])
    d_latent = trial.suggest_categorical('d_latent', [32, 64])
    d_ae_hidden = trial.suggest_categorical('d_ae_hidden', [64, 128])
    kl_weight = trial.suggest_float('kl_weight', 0.001, 0.1, log=True)
    recon_weight = trial.suggest_float('recon_weight', 0.01, 0.5, log=True)

    # --- Training ---
    tech_lookback = trial.suggest_categorical('tech_lookback', [32, 64, 96])
    seq_lookback = trial.suggest_categorical('seq_lookback', [6, 12, 24])
    batch_size = trial.suggest_categorical('batch_size', [48, 64, 96])
    learning_rate = trial.suggest_float('learning_rate', 1e-4, 1e-3, log=True)
    weight_decay = trial.suggest_float('weight_decay', 1e-3, 7e-3, log=True)
    max_norm = trial.suggest_float('max_norm', 0.5, 1.0)

    # --- ContinuousTradingLoss ---
    tc_cost = trial.suggest_float('tc_cost', 0.0005, 0.003, log=True)
    init_direction_weight = trial.suggest_float('init_direction_weight', 0.5, 1.5)
    final_direction_weight = trial.suggest_float('final_direction_weight', 0.05, 0.3)
    init_reg_weight = trial.suggest_float('init_reg_weight', 0.1, 0.5)
    target_exposure = trial.suggest_float('target_exposure', 0.2, 0.5)
    use_sortino = trial.suggest_categorical('use_sortino', [True, False])

    NUM_EPOCHS = 20
    WARMUP_EPOCHS = 5

    # --- Dataloaders (with session filter + stride) ---
    try:
        train_loader, val_loader = build_v3_loaders(
            prep,
            tech_lookback=tech_lookback,
            seq_lookback_bars=seq_lookback,
            batch_size=batch_size,
            sample_session=SAMPLE_SESSION,
            sample_session_start=SAMPLE_SESSION_START,
            sample_session_end=SAMPLE_SESSION_END,
            stride=SAMPLE_STRIDE,
        )
    except Exception as e:
        print(f"Dataloader failed: {e}")
        return -1e9

    # --- Model ---
    model_kwargs = dict(
        f_tech=F_TECH,
        f_seq=F_SEQ,
        f_ae=F_AE,
        ae_type=ae_type,
        d_latent=d_latent,
        d_ae_hidden=d_ae_hidden,
        kl_weight=kl_weight,
        recon_weight=recon_weight,
        n_tickers=prep.n_tickers,
        n_asset_classes=prep.n_asset_classes,
        n_asset_subclasses=prep.n_asset_subclasses,
        d_model=d_model,
        d_static_emb=d_static_emb,
        backbone=backbone,
        n_heads=n_heads,
        n_layers=n_layers,
        d_ff=d_ff,
        d_state=d_state,
        d_conv=d_conv,
        expand=expand,
        dropout=dropout,
        grn_dropout=grn_dropout,
    )

    model = MMTFv3Core(**model_kwargs).to(device)

    optimizer = optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

    loss_fn = ContinuousTradingLoss(
        tc_cost=tc_cost,
        direction_weight=init_direction_weight,
        reg_weight=init_reg_weight,
        target_exposure=target_exposure,
        use_sortino=use_sortino,
    ).to(device)

    sharpe_sched = SharpeScheduler(
        warmup_epochs=WARMUP_EPOCHS,
        total_epochs=NUM_EPOCHS,
        initial_direction_weight=init_direction_weight,
        final_direction_weight=final_direction_weight,
        initial_reg_weight=init_reg_weight,
        final_reg_weight=0.05,
        initial_target_exposure=0.2,
        final_target_exposure=target_exposure,
    )

    best_sharpe = -1e9
    patience_counter = 0
    prev_val_loss = None

    for epoch in range(NUM_EPOCHS):
        sharpe_sched.step(epoch, loss_fn)

        train_loss, train_metrics = train_epoch_v3(
            model, train_loader, loss_fn, optimizer, device,
            max_norm=max_norm, unpack_fn=unpack_v3_batch,
        )
        val_metrics = evaluate_v3(
            model, val_loader, loss_fn, device,
            unpack_fn=unpack_v3_batch,
        )
        scheduler.step()

        val_loss = val_metrics['loss']
        val_sharpe = val_metrics['sharpe']

        if math.isnan(val_loss) or math.isinf(val_loss):
            print(f"  E{epoch+1:02d} | val_loss NaN/Inf -- killing trial")
            return -1e9
        if val_loss > 100.0:
            print(f"  E{epoch+1:02d} | val_loss={val_loss:.1f} exploding -- killing trial")
            return -1e9
        if prev_val_loss is not None and val_loss > prev_val_loss * 5.0 and epoch >= 3:
            print(f"  E{epoch+1:02d} | val_loss spiked -- killing trial")
            return -1e9
        prev_val_loss = val_loss

        print(
            f"  E{epoch+1:02d} | Loss: {val_loss:.4f} | Sharpe: {val_sharpe:.4f} "
            f"| WinRate: {val_metrics['win_rate']:.1f}% | DirAcc: {val_metrics['dir_accuracy']:.1f}% "
            f"| Exposure: {val_metrics['avg_exposure']:.3f} | {backbone.upper()}"
        )

        if val_sharpe > best_sharpe:
            best_sharpe = val_sharpe
            patience_counter = 0
            trial.set_user_attr('final_sharpe', val_sharpe)
            trial.set_user_attr('final_sortino', val_metrics['sortino'])
            trial.set_user_attr('final_win_rate', val_metrics['win_rate'])
            trial.set_user_attr('final_dir_acc', val_metrics['dir_accuracy'])
            trial.set_user_attr('final_pf', val_metrics['profit_factor'])
            trial.set_user_attr('final_exposure', val_metrics['avg_exposure'])
            trial.set_user_attr('final_loss', val_loss)
            trial.set_user_attr('backbone', backbone)
        else:
            patience_counter += 1

        trial.report(val_sharpe, epoch)
        if trial.should_prune():
            raise optuna.TrialPruned()
        if patience_counter >= 8:
            break

    return best_sharpe

## 5. Run Optuna Optimization

In [ ]:
N_TRIALS = 30
bb_tag = "tuned" if TUNE_BACKBONE else BACKBONE
STUDY_NAME = f"mmtfv3_{'_'.join(TICKERS)}_{bb_tag}_continuous"

study = optuna.create_study(
    study_name=STUDY_NAME,
    direction='maximize',
    sampler=TPESampler(seed=42),
    pruner=MedianPruner(n_startup_trials=3, n_warmup_steps=5),
)

print(f"Starting optimization: {N_TRIALS} trials")
print(f"Study: {STUDY_NAME}")
print(f"Tickers: {', '.join(TICKERS)}")
print(f"Target: {TARGET_HORIZON_MINUTES}min forward return → continuous position")
print(f"Loss: ContinuousTradingLoss + SharpeScheduler")
print("-" * 60)

In [ ]:
study.optimize(
    objective,
    n_trials=N_TRIALS,
    show_progress_bar=True,
    gc_after_trial=True,
)

In [ ]:
# Best trial results
best_trial = study.best_trial
print(f"\nBest trial #{best_trial.number}:")
print(f"  Sharpe: {best_trial.value:.6f}")
if best_trial.user_attrs:
    for k in ['final_sharpe', 'final_sortino', 'final_win_rate', 'final_dir_acc',
              'final_pf', 'final_exposure', 'backbone']:
        print(f"  {k}: {best_trial.user_attrs.get(k, 'N/A')}")

print(f"  Params:")
best_params = best_trial.params
for key, value in sorted(best_params.items()):
    print(f"    {key}: {value}")

# Save
best_params['best_value'] = best_trial.value
best_params['tickers'] = TICKERS
best_params['target_horizon_minutes'] = TARGET_HORIZON_MINUTES
best_params['backbone'] = best_params.get('backbone', BACKBONE)

prefix = f"{'_'.join(TICKERS)}_mmtfv3_{bb_tag}_optuna"
with open(RESULTS_PATH / f"{prefix}_best_params.json", 'w') as f:
    json.dump(best_params, f, indent=2, default=str)
print(f"\nSaved to: {RESULTS_PATH / f'{prefix}_best_params.json'}")

In [ ]:
# Save study artifacts
import joblib

joblib.dump(study, RESULTS_PATH / f"{prefix}_study.pkl")
df_trials = study.trials_dataframe()
df_trials.to_csv(RESULTS_PATH / f"{prefix}_all_trials.csv", index=False)
print(f"Study artifacts saved to {RESULTS_PATH}")

## 6. Visualization

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

valid_trials = df_trials[df_trials['state'] == 'COMPLETE']

# 1. Optimization history
ax = axes[0, 0]
ax.plot(valid_trials.index, valid_trials['value'], 'b-o', alpha=0.6, label='Trial Sharpe')
ax.axhline(y=study.best_value, color='r', linestyle='--', label=f'Best: {study.best_value:.4f}')
ax.set_xlabel('Trial')
ax.set_ylabel('Sharpe Ratio')
ax.set_title('Optimization History')
ax.legend()
ax.grid(True, alpha=0.3)

# 2. Parameter importance
ax = axes[0, 1]
try:
    importances = optuna.importance.get_param_importances(study)
    params = list(importances.keys())[:10]
    values = [importances[p] for p in params]
    colors = plt.cm.viridis(np.linspace(0.2, 0.8, len(params)))
    ax.barh(params, values, color=colors)
    ax.set_xlabel('Importance')
    ax.set_title('Hyperparameter Importance')
    ax.grid(True, alpha=0.3, axis='x')
except:
    ax.text(0.5, 0.5, 'Not enough completed trials', ha='center', va='center', transform=ax.transAxes)
    ax.set_title('Hyperparameter Importance')

# 3. Learning rate vs Sharpe
ax = axes[1, 0]
if 'params_learning_rate' in valid_trials.columns:
    ax.scatter(valid_trials['params_learning_rate'], valid_trials['value'],
               c=valid_trials.index, cmap='viridis', alpha=0.7, s=100)
    ax.set_xscale('log')
    ax.set_xlabel('Learning Rate')
    ax.set_ylabel('Sharpe')
    ax.set_title('Learning Rate vs Sharpe')
    ax.grid(True, alpha=0.3)

# 4. d_model vs Sharpe
ax = axes[1, 1]
if 'params_d_model' in valid_trials.columns:
    d_models = sorted(valid_trials['params_d_model'].unique())
    data_by_d = [valid_trials[valid_trials['params_d_model'] == d]['value'].values for d in d_models]
    bp = ax.boxplot(data_by_d, positions=range(len(d_models)), patch_artist=True)
    for patch, color in zip(bp['boxes'], plt.cm.Set2(np.linspace(0, 1, len(d_models)))):
        patch.set_facecolor(color)
    ax.set_xticks(range(len(d_models)))
    ax.set_xticklabels([str(int(d)) for d in d_models])
    ax.set_xlabel('d_model')
    ax.set_ylabel('Sharpe')
    ax.set_title('Model Size vs Sharpe')
    ax.grid(True, alpha=0.3, axis='y')

plt.suptitle(f"MMTFv3 Optimization ({', '.join(TICKERS)})", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(RESULTS_PATH / f"{prefix}_results.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Loss component analysis
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# 1. tc_cost vs Sharpe
ax = axes[0, 0]
if 'params_tc_cost' in valid_trials.columns:
    ax.scatter(valid_trials['params_tc_cost'], valid_trials['value'],
               c=valid_trials.index, cmap='viridis', alpha=0.7, s=80)
    ax.set_xscale('log')
    ax.set_xlabel('Transaction Cost')
    ax.set_ylabel('Sharpe')
    ax.set_title('TC Cost vs Sharpe')
    ax.grid(True, alpha=0.3)

# 2. target_exposure vs Sharpe
ax = axes[0, 1]
if 'params_target_exposure' in valid_trials.columns:
    ax.scatter(valid_trials['params_target_exposure'], valid_trials['value'],
               c=valid_trials.index, cmap='plasma', alpha=0.7, s=80)
    ax.set_xlabel('Target Exposure')
    ax.set_ylabel('Sharpe')
    ax.set_title('Target Exposure vs Sharpe')
    ax.grid(True, alpha=0.3)

# 3. init_direction_weight vs Sharpe
ax = axes[0, 2]
if 'params_init_direction_weight' in valid_trials.columns:
    ax.scatter(valid_trials['params_init_direction_weight'], valid_trials['value'],
               c=valid_trials.index, cmap='coolwarm', alpha=0.7, s=80)
    ax.set_xlabel('Initial Direction Weight')
    ax.set_ylabel('Sharpe')
    ax.set_title('Direction Weight vs Sharpe')
    ax.grid(True, alpha=0.3)

# 4. tech_lookback vs Sharpe
ax = axes[1, 0]
if 'params_tech_lookback' in valid_trials.columns:
    lbs = sorted(valid_trials['params_tech_lookback'].unique())
    data_by_lb = [valid_trials[valid_trials['params_tech_lookback'] == lb]['value'].values for lb in lbs]
    bp = ax.boxplot(data_by_lb, positions=range(len(lbs)), patch_artist=True)
    for patch, color in zip(bp['boxes'], plt.cm.Set3(np.linspace(0, 1, len(lbs)))):
        patch.set_facecolor(color)
    ax.set_xticks(range(len(lbs)))
    ax.set_xticklabels([str(int(lb)) for lb in lbs])
    ax.set_xlabel('Tech Lookback (bars)')
    ax.set_ylabel('Sharpe')
    ax.set_title('Tech Lookback vs Sharpe')
    ax.grid(True, alpha=0.3, axis='y')

# 5. Dropout vs Sharpe
ax = axes[1, 1]
if 'params_dropout' in valid_trials.columns:
    ax.scatter(valid_trials['params_dropout'], valid_trials['value'],
               c=valid_trials.index, cmap='plasma', alpha=0.7, s=80)
    ax.set_xlabel('Dropout')
    ax.set_ylabel('Sharpe')
    ax.set_title('Dropout vs Sharpe')
    ax.grid(True, alpha=0.3)

# 6. Backbone comparison
ax = axes[1, 2]
if 'params_backbone' in valid_trials.columns:
    for bb_name in ['mamba', 'transformer']:
        mask = valid_trials['params_backbone'] == bb_name
        if mask.any():
            vals = valid_trials.loc[mask, 'value']
            ax.hist(vals, bins=15, alpha=0.5, label=bb_name.upper())
    ax.set_xlabel('Sharpe')
    ax.set_ylabel('Count')
    ax.set_title('Backbone Comparison')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle(f"Parameter Analysis ({', '.join(TICKERS)})", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(RESULTS_PATH / f"{prefix}_param_analysis.png", dpi=150, bbox_inches='tight')
plt.show()

## 7. Train Final Model with Best Parameters

In [ ]:
best = best_params
print("Training final model with best parameters:")
for k, v in sorted(best.items()):
    if k not in ('best_value', 'tickers', 'target_horizon_minutes', 'backbone'):
        print(f"  {k}: {v}")

In [ ]:
# Build final dataloaders + model
tech_lookback = best['tech_lookback']
seq_lookback = best['seq_lookback']

train_loader, val_loader = build_v3_loaders(
    prep,
    tech_lookback=tech_lookback,
    seq_lookback_bars=seq_lookback,
    batch_size=best['batch_size'],
    sample_session=SAMPLE_SESSION,
    sample_session_start=SAMPLE_SESSION_START,
    sample_session_end=SAMPLE_SESSION_END,
    stride=SAMPLE_STRIDE,
)
print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")

backbone = best.get('backbone', BACKBONE)
print(f"\nUsing {backbone.upper()} backbone")

final_model = MMTFv3Core(
    f_tech=F_TECH,
    f_seq=F_SEQ,
    f_ae=F_AE,
    ae_type=best.get('ae_type', 'vae'),
    d_latent=best['d_latent'],
    d_ae_hidden=best['d_ae_hidden'],
    kl_weight=best['kl_weight'],
    recon_weight=best['recon_weight'],
    n_tickers=prep.n_tickers,
    n_asset_classes=prep.n_asset_classes,
    n_asset_subclasses=prep.n_asset_subclasses,
    d_model=best['d_model'],
    d_static_emb=best['d_static_emb'],
    backbone=backbone,
    n_heads=best['n_heads'],
    n_layers=best['n_layers'],
    d_ff=best.get('d_ff', 512),
    d_state=best.get('d_state', 16),
    d_conv=best.get('d_conv', 4),
    expand=best.get('expand', 2),
    dropout=best['dropout'],
    grn_dropout=best['grn_dropout'],
).to(device)

print(f"Model parameters: {sum(p.numel() for p in final_model.parameters()):,}")

In [ ]:
NUM_EPOCHS = 30
WARMUP_EPOCHS = 5

loss_fn = ContinuousTradingLoss(
    tc_cost=best['tc_cost'],
    direction_weight=best['init_direction_weight'],
    reg_weight=best['init_reg_weight'],
    target_exposure=best['target_exposure'],
    use_sortino=best['use_sortino'],
).to(device)

sharpe_sched = SharpeScheduler(
    warmup_epochs=WARMUP_EPOCHS,
    total_epochs=NUM_EPOCHS,
    initial_direction_weight=best['init_direction_weight'],
    final_direction_weight=best['final_direction_weight'],
    initial_reg_weight=best['init_reg_weight'],
    final_reg_weight=0.05,
    initial_target_exposure=0.2,
    final_target_exposure=best['target_exposure'],
)

optimizer = optim.AdamW(
    final_model.parameters(),
    lr=best['learning_rate'],
    weight_decay=best['weight_decay'],
)
lr_scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, T_0=10, T_mult=2, eta_min=best['learning_rate'] * 0.001,
)

history = {
    'train_loss': [], 'val_loss': [],
    'val_sharpe': [], 'val_sortino': [],
    'val_win_rate': [], 'val_dir_acc': [],
    'val_pf': [], 'val_exposure': [],
    'val_max_dd': [], 'lr': [],
    'direction_weight': [], 'reg_weight': [],
}

best_sharpe = -1e9
best_state = None

print(f"Training for {NUM_EPOCHS} epochs (MMTFv3Core {backbone.upper()})")
print(f"Loss: ContinuousTradingLoss + SharpeScheduler (warmup={WARMUP_EPOCHS})")
print("=" * 90)

for epoch in range(NUM_EPOCHS):
    sharpe_sched.step(epoch, loss_fn)

    train_loss, train_metrics = train_epoch_v3(
        final_model, train_loader, loss_fn, optimizer, device,
        max_norm=best['max_norm'], unpack_fn=unpack_v3_batch,
    )
    val_metrics = evaluate_v3(
        final_model, val_loader, loss_fn, device,
        unpack_fn=unpack_v3_batch,
    )
    lr_scheduler.step()

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_metrics['loss'])
    history['val_sharpe'].append(val_metrics['sharpe'])
    history['val_sortino'].append(val_metrics['sortino'])
    history['val_win_rate'].append(val_metrics['win_rate'])
    history['val_dir_acc'].append(val_metrics['dir_accuracy'])
    history['val_pf'].append(val_metrics['profit_factor'])
    history['val_exposure'].append(val_metrics['avg_exposure'])
    history['val_max_dd'].append(val_metrics['max_drawdown'])
    history['lr'].append(optimizer.param_groups[0]['lr'])
    history['direction_weight'].append(loss_fn.direction_weight)
    history['reg_weight'].append(loss_fn.reg_weight)

    is_best = val_metrics['sharpe'] > best_sharpe
    if is_best:
        best_sharpe = val_metrics['sharpe']
        best_state = final_model.state_dict().copy()

    marker = " [BEST]" if is_best else ""
    print(
        f"E{epoch+1:02d}/{NUM_EPOCHS} | "
        f"Loss: {train_loss:.4f}/{val_metrics['loss']:.4f} | "
        f"Sharpe: {val_metrics['sharpe']:.4f} | "
        f"Sortino: {val_metrics['sortino']:.4f} | "
        f"WR: {val_metrics['win_rate']:.1f}% | "
        f"PF: {val_metrics['profit_factor']:.2f} | "
        f"Exp: {val_metrics['avg_exposure']:.3f} | "
        f"DW: {loss_fn.direction_weight:.2f}{marker}"
    )

print("=" * 90)
print(f"Best Sharpe: {best_sharpe:.6f}")

In [ ]:
# Load best model state
if best_state:
    final_model.load_state_dict(best_state)
    print("Loaded best model state")

In [ ]:
# Plot training history
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

ax = axes[0, 0]
ax.plot(history['train_loss'], label='Train', alpha=0.8)
ax.plot(history['val_loss'], label='Val', alpha=0.8)
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
ax.set_title('Total Loss'); ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[0, 1]
ax.plot(history['val_sharpe'], 'b-', label='Sharpe', alpha=0.8)
ax.plot(history['val_sortino'], 'g--', label='Sortino', alpha=0.6)
ax.axhline(y=0, color='gray', linestyle=':')
ax.set_xlabel('Epoch'); ax.set_ylabel('Ratio')
ax.set_title('Risk-Adjusted Returns'); ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[0, 2]
ax.plot(history['val_win_rate'], 'g-', label='Win Rate', alpha=0.8)
ax.plot(history['val_dir_acc'], 'b--', label='Dir Accuracy', alpha=0.6)
ax.axhline(y=50, color='gray', linestyle=':', label='50%')
ax.set_xlabel('Epoch'); ax.set_ylabel('%')
ax.set_title('Accuracy Metrics'); ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[1, 0]
ax.plot(history['val_pf'], 'r-', alpha=0.8)
ax.axhline(y=1.0, color='gray', linestyle=':')
ax.set_xlabel('Epoch'); ax.set_ylabel('Profit Factor')
ax.set_title('Profit Factor'); ax.grid(True, alpha=0.3)

ax = axes[1, 1]
ax.plot(history['val_exposure'], 'purple', label='Avg Exposure', alpha=0.8)
ax.plot(history['val_max_dd'], 'red', label='Max Drawdown', alpha=0.6)
ax.set_xlabel('Epoch'); ax.set_ylabel('Value')
ax.set_title('Exposure & Drawdown'); ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[1, 2]
ax.plot(history['direction_weight'], 'b-', label='Direction Weight', alpha=0.8)
ax.plot(history['reg_weight'], 'r--', label='Reg Weight', alpha=0.6)
ax.set_xlabel('Epoch'); ax.set_ylabel('Weight')
ax.set_title('SharpeScheduler Progression'); ax.legend(); ax.grid(True, alpha=0.3)

plt.suptitle(f"MMTFv3 {backbone.upper()} Training ({', '.join(TICKERS)})", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(RESULTS_PATH / f"{prefix}_training_history.png", dpi=150, bbox_inches='tight')
plt.show()

## 8. Model Diagnostics

In [ ]:
from CTAFlow.models.deep_learning.multi_branch.tft.mmtf_v3_core import print_v3_diagnostics

# Get tracker from a val pass
final_model.eval()
with torch.no_grad():
    for batch in val_loader:
        inputs, targets = unpack_v3_batch(batch, device=device)
        position, ae_losses, tracker = final_model(
            **inputs, return_ae_losses=True, return_tracker=True,
        )
        break

# Evaluate final metrics
final_metrics = evaluate_v3(
    final_model, val_loader, loss_fn, device,
    unpack_fn=unpack_v3_batch,
)

print_v3_diagnostics(tracker, final_metrics, epoch=NUM_EPOCHS)

In [ ]:
# Position distribution analysis
final_model.eval()
all_positions = []
all_returns = []

with torch.no_grad():
    for batch in val_loader:
        inputs, targets = unpack_v3_batch(batch, device=device)
        position, _ = final_model(**inputs, return_ae_losses=True)
        all_positions.append(position.squeeze().cpu().numpy())
        all_returns.append(targets.squeeze().cpu().numpy())

positions = np.concatenate(all_positions)
returns = np.concatenate(all_returns)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Position distribution
ax = axes[0]
ax.hist(positions, bins=100, alpha=0.7, color='steelblue')
ax.axvline(x=0, color='red', linestyle='--', alpha=0.5)
ax.set_xlabel('Position')
ax.set_ylabel('Count')
ax.set_title(f'Position Distribution (mean={positions.mean():.3f}, std={positions.std():.3f})')
ax.grid(True, alpha=0.3)

# Position vs return scatter
ax = axes[1]
ax.scatter(returns, positions, alpha=0.05, s=5, c='steelblue')
ax.axhline(y=0, color='gray', linestyle=':')
ax.axvline(x=0, color='gray', linestyle=':')
ax.set_xlabel('Forward Return')
ax.set_ylabel('Position')
ax.set_title('Position vs Return')
ax.grid(True, alpha=0.3)

# Cumulative strategy PnL
strategy_ret = positions * returns
cum_pnl = np.cumsum(strategy_ret)
ax = axes[2]
ax.plot(cum_pnl, 'b-', alpha=0.8)
ax.axhline(y=0, color='gray', linestyle=':')
ax.fill_between(range(len(cum_pnl)), cum_pnl, 0,
                where=cum_pnl >= 0, color='green', alpha=0.1)
ax.fill_between(range(len(cum_pnl)), cum_pnl, 0,
                where=cum_pnl < 0, color='red', alpha=0.1)
ax.set_xlabel('Sample')
ax.set_ylabel('Cumulative PnL')
ax.set_title('Validation Cumulative PnL')
ax.grid(True, alpha=0.3)

plt.suptitle(f"MMTFv3 {backbone.upper()} Position Analysis", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(RESULTS_PATH / f"{prefix}_position_analysis.png", dpi=150, bbox_inches='tight')
plt.show()

## 9. Save Final Model

In [ ]:
model_path = RESULTS_PATH / f"{prefix}_best_model.pth"
torch.save({
    'model_state_dict': final_model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'best_sharpe': best_sharpe,
    'params': best,
    'tickers': TICKERS,
    'dims': dims,
    'n_tickers': prep.n_tickers,
    'n_asset_classes': prep.n_asset_classes,
    'n_asset_subclasses': prep.n_asset_subclasses,
    'target_horizon_minutes': TARGET_HORIZON_MINUTES,
    'history': history,
    'final_metrics': final_metrics,
    'architecture': 'MMTFv3Core',
    'backbone': backbone,
    'ae_config': {
        'f_ae': F_AE,
        'ae_type': best.get('ae_type', 'vae'),
        'd_latent': best['d_latent'],
        'd_ae_hidden': best['d_ae_hidden'],
        'kl_weight': best['kl_weight'],
        'recon_weight': best['recon_weight'],
        'ae_features': ['return_1d', 'return_5d', 'return_21d', 'rv_1d'],
    },
    'loss_config': {
        'type': 'ContinuousTradingLoss',
        'tc_cost': best['tc_cost'],
        'use_sortino': best['use_sortino'],
        'scheduler': 'SharpeScheduler',
    },
    'tech_feature_cols': prep._tech_feature_cols,
}, model_path)

history_df = pd.DataFrame(history)
history_df.to_csv(RESULTS_PATH / f"{prefix}_training_history.csv", index=False)

print(f"\n{'=' * 60}")
print("TRAINING COMPLETE")
print(f"{'=' * 60}")
print(f"\nArchitecture: MMTFv3Core {backbone.upper()}")
print(f"Target: {TARGET_HORIZON_MINUTES}min forward return → position ∈ [-1, 1]")
print(f"AE input: [ret_1d, ret_5d, ret_21d, rv_1d] (f_ae={F_AE})")
print(f"Best Sharpe: {best_sharpe:.6f}")
print(f"\nArtifacts saved to: {RESULTS_PATH}")
print(f"  - {prefix}_best_model.pth")
print(f"  - {prefix}_best_params.json")
print(f"  - {prefix}_study.pkl")
print(f"  - {prefix}_all_trials.csv")
print(f"  - {prefix}_results.png")
print(f"  - {prefix}_training_history.png")
print(f"  - {prefix}_position_analysis.png")